# SciGeoGuard-X — Multi-Model Blind Validation v5

## Two independent LLM judges + third-model blind adjudication + targeted human audit

This notebook is the next validation stage after SciGeoGuard-X v3/v4.

### Core protocol

- **Judge A:** `Qwen/Qwen2.5-1.5B-Instruct`
- **Judge B:** `microsoft/Phi-3.5-mini-instruct`
- **Judge C:** `HuggingFaceTB/SmolLM2-1.7B-Instruct`
- A and B evaluate the same blinded Earth-observation workflows independently.
- They never see SciGeoGuard-X predictions, generated labels, family names, or each other's outputs.
- Judge C is run only on A/B disagreements and remains blind to A/B answers.
- If two of the three models agree, that becomes the **multi-model consensus**.
- If all three disagree, the case becomes `UNRESOLVED`.
- A targeted human-audit file is created from unresolved cases, system disagreements, and random spot checks.

The notebook uses public local Hugging Face models and requires **no paid API key**.

> Multi-model consensus is not human expert ground truth. It is reported as a separate validation layer.

## 0. Colab runtime

For the full 180-case experiment, use:

**Runtime → Change runtime type → T4 GPU**

Models are loaded **one at a time**, evaluated, saved, and deleted before the next model is loaded. Results are checkpointed so interrupted runs can resume.

In [ ]:
RUN_MODELS = True
MAX_CASES = 180
CHECKPOINT_EVERY = 5
MAX_NEW_TOKENS = 140
HUMAN_AUDIT_TARGET = 50
SEED = 20261002

MODEL_A = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_B = "microsoft/Phi-3.5-mini-instruct"
MODEL_C = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

from pathlib import Path
import os, json, math, random, re, hashlib, gc, shutil, gzip, base64
from datetime import datetime, timezone

import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/content") if Path("/content").exists() else Path("/mnt/data")
WORK = ROOT / "scigeoguard_v5"
WORK.mkdir(parents=True, exist_ok=True)

print("SciGeoGuard-X v5")
print("RUN_MODELS =", RUN_MODELS)
print("MAX_CASES =", MAX_CASES)
print("WORK =", WORK)

# Part I — Embedded blinded cases

The 180 workflows from the v4 reviewer pack are embedded directly in this notebook, so no reviewer CSV upload is required.

In [ ]:
REVIEWER_PACK_GZIP_B64 = """H4sIABRsv2oC/+1d25LbRpJ9n6/o6NcVNXXB1W/aUa9WMZLsmNbau+FwICASzcYKBLAAKKk9Mf++hQtJXAokwQYbAHkUlsfTKLLJqsysvJw8GTnfXOe7FUQLJ3o1t2PHchevvgfR1wcv+G65fuL4ySs7StwHe57E1v/Ggf8qTpyw+M8oe7kTWd+caOHOk90PnCgKImvu2XG8+2HsiHVu8rT7yTzwH9yF488di1pJYKnlR6uV+O2ll7uLv9BX9+/+m1v//uH9p7eW+IzBwlo4YfJoEUpf3f7z9nYZ2N7t7U83t7elp7e3r8QPIuf/1m7kLKz/W9t+Ij5Gus5fe17l4dp3k/KDhTN3YzfwrcRdOeUHK/uHFTlx4K2T9PFK8m6x9cWOHMsRG/iYPn6wvdipPJ97wXphrez4a+V5+t5rsSlRYoszqHzQxFmFQWR7Yrc8J7LTnXuUfpFvtrd2LLFiuf3Y/7p9dfu72CTfzr9K+m5RJH7FzCT5HpW35vbW8Zxvdvrt8oebrRGfL/9BPLe94o3EQSauvw7Wcf5oHsX5g7tf7t/9xJmm8M0pVLfMJJVvVXpc+VphFMydOHb9pSU+lOOVn8XrSIinY62ChVMc/tv7j/mvEwKXuOJjWgs7Wa+Kj/TuIyPEyBfYc7FfsZvIjtj+Zrue/cVzGk/Ex/nm+Onu52/5tyB0It+dr+Obt3fFr/5i+wsrCrzKC/1AfBBbaMIq3eh5XH7WcuK2J3Rxc6CVFwRfYif6Jh6tXF/+c/tH5Z2SJD+Wf/7rX69uKoLw3RaiMCt2UiYL2YLN3lelYd6jOFDy+gwCsXkgk4bf3t0biuV4nhvGgbvoUyoenxaRvfbc+U3xScYvFn8UJiIINwe5+uL6jrXZufw7hHZkr/KXiJen/5sd2pfsBAvZ2r2Dn764MKzNVxeXS/bS7OSDrY3JPk725y+s1fBzBYb/dMNPBzP8FIYfhn9Iwy+kwdRg7kdi7r+uWMXg87LBdwJr5aVf3V1kZskiilmz+vUlMP1tpv/u51lsr0LPiWecKaZM55P0bkg1rFhYVfyttk1Q7bdK0IvG2+vkMRARrZC4b85MPF2s58kU9T4OPam6pju6cRjyA8oXZQJiOf4if8gIU2ZEnxH6mZCfsn/+Lft3sdqJEytOhEqUl2uy5SULoOQW4D/vPrz9+b8+z/7jzcf3H/4n01VrIc7FzzR05cYrO5kLP5CwmkVoWQnD0GYYFq44I1/uBVSfnckhhCkYhQtw9AVOyhf4vBqwqc2ALXJWQeKIr+xnZ0eoLo3cqsugrm3qmu3WLP39MyaN4vLtdH6kSdyDN7hw++zoqaPKMijsSyusOJ2t3Fl2KO5tZ7GVv0Z0d09lgvHFnn8VR5+IwO6cnt1YZKPkn6ifqcQ/uQIrX5cLJpOLyHnwnLnknr8CuWDsKuWifvsXV1DlctvvAzTi/1WYXpzp9R1EcVsCgW4TCCx3IioXImW1dxXvaB/jjKwccQ5FULJ0hHjE7jx/XbBOwnVi7TzXesJBKzssnji0uTjiyJo/prewcFZ4zVmpL4Gj0uao/BZE3uJv6Va1OCrpVlrZXh42O3biLIWmbXNTVbujCMNzea7K3f2bm90mntus/J5a6HS7uPiriL+q+KuJv7r4a4i/acnWVF8JS07+6N/+5Cmnrsr+xfVcX+hAkXCwo6WTNDRHy17mfCsQB8GD5TspyuEhiFZF0vCnmyRaOyWzoB+OY4iKOKafOIaeM45ptw0UYcxYwphc+xDF7PVWKaIYRDFyudAQxYwmiuEvH8QY0ppJPHd84e4IufLsUAii/dVOQxpSr6FK18FxaXNc5o9uGMsskLtK9xe1U9ROS3q1KZSVVewhEnpfxB3iMLlZUmXzQD6CIh/x/HwEHSwfQZGPuOR8hBCh+Z9B3Hs6QuxFzSzEnni9JV7jzO04Ebe6LFGZLbrZLOrVNGz8T2NG1c9UkkW/CJvx/uPdP97NHhyhUJEUIJECYsTHamAkV1eHkCgJhCaFA9XNyac3wp68++XjTbbJE4xD0UVzrWDq+nURCoPizmXeYGE68huN/FELDTcBa9kM7sfOlu+Eoivu7d2vdx9+/uXj3afP1sM6/W3ly0SCkMsSq+e+FdKsxKXeCpnW41LodCm0YEQvNjzsYBJ2Gs2aGl2KNLLG3CgMvI1i112+1rUICttUeRs+o0Y9nXTOJEPDtEztbK7bPkND3jQaaz9eh0KaEmdR/h3LFL1PCGkg5ltXw3C0+wDxtgJZNRkbXRTKIPza5Ol8mWAYjlEUnYaGrhQY4J1FUFq7phWOrml0TW8D/c8I9NE1DbqMSfdPV4AA9Jj+KwO4xX5wi3yg/isOvw/AxekA1CgBcBHARYlc8GuVi3ECF5nSA3Ix9Gx/E9QexC1SbT/dC1VB9/J8uhddJWB7QaaqP8Qi1blR1mJ9P2ZRYcAsArM4hPJufopSVOWOnq+/bFoLWvLO/JRClNEsRMWhWC1ONzM9pRaEeglKug5moc0sVG5s3Oq41fff6rlyFfe6k67anaLKDSNXmNDxc23PhK+wA7pauenNA/46h7/+fH9d1cHOCM3uQbMJaddsSo2yZjMibSFcufnp7vZYXN71FqTmIqh5m5qv/a9+8N2fuX64TjpUk6HeG/UOHh7cuRD6iWn2739sNI1KNa2M3Vw5tm/Zy2XkLNsw3PuWQ/sOoj3PG0rjar2YEHujVo7VOPPj4+2Vs3BtvzrYgh3ZvKGheQPNGyNo3qAGmjf2Nm8wvj8LzlRkwcEkiCz4hWfBmXKoU5+xoTr1CTr1ca9XBOLKOvW1wQD8Gjr1r7dTn6ntYwxNNOSgIQf6PEBDDmYYoQHn+Q04f6+m9bSW6WXbvdmW1Jo8nKkyZnbL9m429vsS7gChHGr/d8Cj/acdLWQ6nz9Jy5Ff7LTLtt6CnY41K8bDeUIZYQbOnr0T+0LpC5XX9P0QFc4AUXk+RIUZBiAqgKj0SIKrmHr5HjUOaDGBFj9fi6nKoMZQ4165rA1WVuMDeFFGocY99HcxCjWGGvc0zlsRzl2Hcd7qgXHenBwgtDdQFkdzGMriF09hz2lXnkIDPIXgKYRzMAmewtda5Y9+Am0hZ9Lcefj4FGcHnDevfHN3/Mf1BHrbUliHNuvg+gvnh8w4fHr763tkzadmH2bkta5yQ9M1puiGZqqUycwFeW3onCmawUzdFP/Qgyj4OE5rYxWROsKKFB+KFiJdh0VwfnzDuIaGcTSMQ9FH0FZqcKq2t5Uq1CBlFT+MhqVAwwINOw40rPqZaIfQsOfPGmBSFfByl4V/5UewEhMOVuIJsBK3J4JBSgxS4gmRz6YtrSAlBikx5GLcpMSUvTQpMdeaaans5JaRm25svLKT+aOEk6W5CP5Ka1nKWUzR7NzefmBviqho7M5KaZPHevcEyaO8NvkicqBdixz4bjQyOWjmJMVpR9nY0m1BqmLeU8PSMPGZ+FiN1hznR/pu4gX5c8k+iBWeO3cTy/bcpb/KQiJJQVKXFiSr+MddiaLO7yFdhxuh7UaYP7qhtEDhrtL9RXUC1YlOUGTVVMq6fKChgKEtqAckssFUhhojtLg/LdYI4WU1NrtihBgwQsAIQa1LGCEpJIgNAwFSSDPXIt577dmpX+/YEgUuP4bS7q8HhYH3tNzYtheuB1Eo7PD38NmTpQrteCFTggsZFzL0uwLaNRjTFUPVNd2kOjUVOWhXV01OFI3rXKw2VWWgK5sdWR4hKI+gPILyyGTLIwzlEZRH9pRHFH4Y1UcpUH39oPrYOVF9msnRujl2VF+hf4D17Z0prwLWB1gf5OIqYH1LR4hHvBnicDhXpTQj17oQShoRmnIKd0XurgRhiXx3EMPz+jkxy+efEbv2b2c2m9mQgH0GZ6ez6oGpSgz0UZiqNIy23t2/udlt4kWPQTwnmdQpU5YUCUa/2OU/nSiQZJ9LT2ES9s1GRLf4ULmG7STtzTYPeHs/uJ5XaNQxOp99mKxsZIo/6U8yNXTjbSXYe8rBlDJl1o/yyyn88uv0yyP7u/X2E1zzsbnmRutUK/nUYky1wlQrsHScearVHC4Zxlo9f6zV11U1dWoervVy1HrB4IJaLxhcXoqpA6VelHplcmGg1DumUi9/+UqvKmkrelin9HblvLKIU3WZy3J24lF6ucSjWQAL3tFuvKNCHtRDvKMXZJY6cFDuNJoeQSFJEIAgAEEAggAEAQgCkGHlQkMAMpoARHtpBkmV7ScbUjjIhnoYe6oqIBsCyUFPY0/1GTM7jD3VD4w9VY9ojlM0xCsTaI7DYEN0x11KFxRBxIKIRSIXIL2/8ohF2R+xcA0Ry/MjFs00EbEgYnn2CEaqqrR9BCNnrMx8rKpShrbS97eKtgNJ72uKpMx+se3dbNT0ElRdqIHav6o/2n/akZS4KX+SdjV8sdOB2vVp6+Bpe2mFF/vCOTuryv/+x0YFtXakPANS/nSkPB8MKc/Pg5T/DKQ8kPJHmNlMGkytT1kAXP4ZcPm/V6Mp/QhmNAXJ336Sv5SA3xq5X+R+kftF7he5X+R+T4DLG8eniBp+C1JESBFdbIqIGMoLpYjMZsfK2o/XoTiSlKu5dCbLFL1CSF0P96xGGNHerhIncor1jUALiRIGrq6M/ToFUL8xXP6nMZylptp3NoXWFoqz4ogPcJzVCdY1SRdbiWhPvFzIbrgbdVe3CK1rYQ/a7MGWPbIlq/ASlIhILFwDM2JqM5xNi2mL2TBPYEbUjmiUYypyj8g9wmNAoxzI95F6HFYuOFKP18zUobE6v3vsCXfH2lBwSJyV7aKz83Sol8vT8f7j3T/egaijI1GHIhWIqlXavNGkLqYNoEobDFClgXp0auQslbtic4OVDd9+/Ez5FuD72w+ogfaD57cfGFynaD9Arru3hmneoWHaONAwrSktNWg3sbaFz/J4ad6of0lXwjK0WYaFG5dizqpJqD47E9c4bMEoMLNHI17JHoJoTT1qhgfDDI/rnOGBufDjG+ChaQdoigi87ud73Vw3DXjduGmPafqdO76z8bDEf1op0MCzQ+shEupb1HvJa9UwtLIa6/tHZDZIjjEiEyMyMSJzSiMylVOAIMYRQBAwJoOBDDc7utBejhuXAgoCKIhcLgAFGQ8UhLKXpiDTzAMUZBTZiOdnIxgnOrIRyEb0l41QdE0tqbEuaVoR0rn27MhKbYgkH1F+DCXeH2yEgfe03Ni6swUbCoKN0Sru2S9inR6CZDbYQF8MkkkByQQksy4Q/KIhmRfHcQdI5hQgmTrrQBtLwQkCTpCroY3NZeD8nCA6l/T/71LP7erXXISwqk357tlsi02SM4AAXnXZ8KoD9Zw+8rO7W1U52O5moN0NsdU42t00Kb7/gmIrc7DYykRsdcWxlXroFqAEtwBugek0Pd/efnpz/+bm3S8fb7JNniAg42LvBMyUmMSdoB26E5ThiDAo7gTcCWWBYJceGWCyEG6BIW6BA708HL08PfTygNsVvTzDkUGzE5p5dOOQd0gGyxgwAu8Q3mFVIIgJmrQp0aTBO5yEd2ju9w4VCu8Qnd5DeIfnt+aT9Afn6y8bJtwWZ1A7wRk0yAEzwGEGECQiSJysUeCnGAUqxeuu1l7i5p2g2dmVGRTr84Ja18JetKIHh+zuf43+fvT3T0oyTHT4H9+gX5pS39KgvzP+bH/vPNPRO99D7zxD7zxacHtl8hNnXVZjScNHHIrtEmeS/U5xmPZXOyP1M2saLV0HtW5T64qqQp2hzvvVOVeuQqGddFUpWjNNlitM6Ph5xJYJX5HTU7hSVnGlqeLizcRix/YtO7bs+XwtArHNTNd6Kqd1LVR9bw1PpuhhJL53mAmw+HLpzjareH997Fi72ZfWIQcmPYsLAdp+bm3f7r69XEbOcnvqxydwUvUrpCK7IazFurAH5VMUSuXZdVVMT6Oss2XLoB6YgQM27h58eKqaGi59mIHnXvqqkWfp5Zc+VfRKeK41L/2HdVolLid1JU59RukEdPfpmp+hNQDf6QDf0WecHtXwczGWoAOUY6fRurTaEj4+xdlh5LbgmxtsHHlad+TbluIWb9NlV9jZHzIl/vT21/fgRZna5T0jr7lmaITpikp1nXNqKjIVJq91SkzVICYVPrDBOT/IkBfH6USrilAd4eEXH4sWQl0HXxlH8Ok3WrTAp38inz4/J59+e4zOod1joV/J8Q1g099XU6UE1Xaw6cvlAmz642HT5y9Npm+YR2YcFGQckHEYvJ2cHtEwdOUZB1PCq7/243UoTjhxFuUjXqbTupuskHtWIyRpV+d461XK6SGFHIbiczydz8NQEZSMwZE4Y0swZXnN8ADke8dIWdgEKiGI3XUdiJcL2Q13Ocj69O3WtbAHbfZg20qDNrHp2INJtomkRsPZuOBtjSJq90YRU07svhWnHEC4wSsSWncj5AthMVo9CGceRAs50jBtshES5QJvALzB9oAKjU8DRcvxF+VQSZtRKmXeSoSlsGJhJZJG00V9edkU8E7+A9HgP/TnP6C/FP7D0HPF2Sn+g3JEHRRzxXuqg9KB5orDRmCu+KQqXioqoaiESuWCMVRCR1MJ1V66EmqqHesmDaQm6iYTrZsAzHXpdRP9tLKJ1mG0JcFoS4y2hF3odbCl2bV7gujonkD3BFSv0j1BuKIRg3JTY1RXiWbKuycMphoqNxVdo6ZumOow3ROm0anq2OA9QNURVUfcssNUHfUZ4X1WHSXY5Lf3H1NCk7efP0pc7t1D6HubvhfM9BhaBLU/hWW+PKOaEnJ8fNzAFCI+Rnx8mTe32Bdm0JeJkCmh3fxlAn8Z/jL85bH4y3qP/jIl7MhmPoJmPjTzjaGZT0Mz395mPkr4/hEuzMAIF0xywiSnaU32PGWYEyXKkbe7gdsdt/sIbndKcbsfuN3VQ9N6qTbUtN4UKItpvdDoikAYmNY7pWm995jWO4FpvZTIcYbZCId4vbK+u+KSXCc7+nhCVNm4B9laxH9tl0A+1WG2G+cw+qEPFHnd8Q99SPublztxaZn8QE+Z/ECJ3nl2p4HZnZjdidmdmN35MpKhSHP6mN152uxOrWL8JbziCydMHiVjO0tPYdcPQOFm5mBhn4mwbwxm/Lvw1qJZsZMyWcgWbPa+Kg3zCWeEbm9/e3dvKJbjeW4YB+6iT6l4fFpE9tpz5zfFJ5mkDf/i+o612bkjjHijFdpPX1wY1v0Zgdz+y5uTKTHbjb8C43+68aeDGX8K4z914z9p238GYYDNf4bN/3vV4lM5uL46JXw75rsBr5euw3XQdh3MH4UHJNN+d5XuL4Z9AtZb0quN2S+r2EMklLqA7ojDNPSKMtMDM3t1zOx9/sxezSAMM3uhxv2pMaNGRY3llNjVcylXX+q3cstKaPdI21Uvwiu/lGpLFohdU4R2xuzcJdVZusdrJYvO5RX1om15t22SrqnmItjxNju+9r/6wXd/5vrhOulgzeGcTQlXWelMppJ+hTh05km29eIrSEaPVZ5Dm9qbkBdTJO+9vf3A3hQX8thjnNImj9UfCpJHuT8EOehTDuLvbkS1kYlCM+ItDGeJvq/iBKW2pYFbzCTIamSknR/pu4kX5M93W+G7hbyJFZ47dxPL9tylv8qmCzQ71qjagT/VAD8M+GGuhR+GUu2lvDBtfwO5oqOBHA3kaCCfVgO5ckoDOdVbMUScAUMEACkwRMAQAUN0UbhRauwHHigAHvQAPKAa1wA8QDWqL1pA2istIDW7TrZSMdnqMiZbqTAKFz7ZiqrkhMlWlJGuneSUoJMcneToJEcn+ctIBv9MODrJ++okp6xi/WnTIxSWa+3ZaTXQsSUEQ+XHMPD7B7KHgfe0DPwzD2RXMMx0tDp79mnFlLHWZD5FMh8NwUjmI5k/oCVHFr/PLD7jnXqBCXqB0QsMH240TYSqWVFmpem5Rc4qSPIQL0NDKqbUhasugwbvD8PS39+CzOopDNNMzi8NljVZDRaHsxU7yw6F9jqLHQyqnm8bMhM7vmxbyu2OPOzQedhRygVjyMKWGN7LV9uhSbm1YOCoNC7dRhNMksZlSu1dT0k0LR0hHrE7PzrVpLZzz5lINSHVNNQYbqSagBtFxukM3HNM6zZUWsNQaQyVRrw6CvSoNqO8T/Qok88cCR+f4uxQM9G2vrmB1zZdum0p7EGbPSg1cVetwae3v75H++7UbMKMvDZkNoG+5geBBHGc3ucVqTki2it+Ly2ktkF7xNpnSSgUIR1aARHSIaRDSHdJIZ3ZkbqUgLoU1KWgLgV16amyYGqQhP4JSzlpQvdL5DFWOn40CrfBeAML1roWhrzNkG8ZkUDzMx3VnST3j5Ck+Z9BvLfhUzuF+ofTLgBSogBACgApUndjAZAqRKkoMzsMICU6AKT9AEjpMABSCg0GgHQ6QEECACkApHK5AIB0RABSbQD8KOfd0EQq0ERAE8F5GQmaSAIPegaaiCv76cgb3AWgIwcdOejIx52SPImNnKtdWSkVsFJeBislihkXz0pZuIBdaSm51qlOoaFOgToFLMFY6hTc1CrKrB9Rp2CoU0yA6ALAhAnUKYrbFIUKFCpQqOgsF9q1ysUFM110o1TlRjMjUZzSn04USMZVl57CR2nPOMhB8OnPxadp4p5BotmjSUmbAeOdEz2oZXlwPa/QqGM0Ofsw2/+XqaAbbxuPvad8SpU0t2jurzJwVBlQZRhIqe/u39zsNhEw6BNrDuopNQdF0jwRCSG0Vo7tW3Zs2fP5erXeMhnUKw6ta2Ev2uzF+493/3gnsxNhJL53mF1u4sulO9t0BP762NEVaDcZQr7JfqtBXqvIVJzbBdjuvr1cRs5ye+rHG4e0aWm5E5fcwliLdWQ3jlJolmfX9TE9krLiVuwDPZyxpMhYTgFZvc8SIGMJZDWoeZGwnLZc6EhYXhqyulu+UmH7sxwNIDWyHMhyIMsxKSDVSdBKhTfTHNnmLyN3Ueb4qdczmotgItq7LBZT9Edubz+wN0XoPPYoprTJY3VKg+RRjqaFHPQpB74bjUwOmiA6cdpRRqqz5W6t3AWpYWncB5n4WA0ON+dH+m7iBflzyT6IFZ47dxPL9tylv8pyJZKLQGml+eQcNJ+g+QTNJ2g+QfM5ZZrPVS0n0D6rR1Fh8TGrB2OhYfFh8S/J4mudCEEbDTcgBH0GISgfLDnMUcW+ghxx3oK3Jz9snpQf1o/hgl/7X/3guy9JE8sXwl6ACR5M8GCChyy8FBe8YnSkUCEmKFQug0IF7t+lU6hoJxGoKGY3rkUGrkVwLcISjIJrUe+Za1ElXRnWGNyDy3APVBiFi2dYO8k/UGnTJmyhK+nOSnyCynPoPQCBAAQCEAhA4BQBgSprhYcwAnjI6fAQPhg8hAMQOHV4yNdJJ/h/e3dvKJbjeW4YB+4COJGR4EQa/YIq70SuTEGuDHJlJAHGQq5MKqqsHA7jCcJ4hPEI4xHGj1sOnMXGi0Ao3ymUVw9zVTFwVfXEVcWHmQIMmAfY9SdFSkRAVgWyKqlcYAzwiMiqKBtgDrCq1fmqYs9dONZDEDlzO04swhUJY1W26GazqFdvZSOgRgqloRIBvRwK3tmDI1Q2csDEv8+XqQgEMZsCUbdLn97cv7l598vHmxLP8aQuqotltEB/c8ebKRQGxZ3LcpWF6cjb6Mgftatjc6OVzeD+KkXlUtAPXgpkqEshpV7FpYBLYScQilQgXnqQ9BmvAW2wa0DDNXDN14Cxn8uW6uCyfT6XLR2MrmC6pPvnN+eXy16rncJOoHZrYaMqWtjQwoYqxyha2LQZU/psYdNIO16ZAq8MAlP4+aCz68xHYmoAKY8GpFwz+LTV4FMFBh8GHw0qaFCBxZ+wxf97zeAzGZJd2KRtAL/rSGk2pkvW4SpouwoqLSe1QD+9I1JNQ1sKov2KchWG30lXlU7RYHquMKHj58m9TPgK1KqmVJL7mrzxLJumHa9X1ndX7Oo62Q3xbap661qoe5u65wO2Z7tRyZi/Db1//vzteL06w+RtrTKpyAmslZfuhLsoDAKvM1fWl8AOtNmBu59nxa0+07mu4vqHGeivK1VVK1qsSi96ofKJm+OVsy0ujaBsJHVa10LB2xR80B6U1yPsNkjB3OhCGboLZZSSwdCH0qmNpJT/kbSRMKVi/bWufKMEfKOXwTdKCby6CyccVQuD3pVxVNMP96krFH3q/fSp03P2qU+i6fTqG9VzPCX61BEhoE+9s1zwa5WLcfap8wHa1DWjFYJEdECQMEIZECRAkDBD+ZJmKGvm/u5DRUH3IboPh4lO7+7f3Ow2EVOST5mSzE7pQ9TJfkhCo3QJSMIJkASVMwWQBCSvn4tI1AxFa0ckUkMzK7pNpUCF0gakuIM8OU3ryenUDc9+s+3dbPT0EnRd6IHav64/2n/akZRSO3+SVqe+2GkJol6gEjsjNsRPt0YYgPjaaGcG0HixL1xVzqrzv/+x1cE9s4g05FnQ6oU8C/IsaPW6pAlEOt+fZ2nMHEKe5YQ8Cxssz8KQZ7nkPEuKEXI2bJt9Uj7pymGMEFeBEeoHI8SGmWXBEHNhlgVmWQAjhFkWwAhNHCOkq81WB3H/rj07nfjl2CKUqTeslh/DTdnvpoSB97TcZKpeGsqMmVsjMC2nKHEoQtxNE8MRKqztL+8SlHd7KO8aqspR3oVC90Mvq8wo70Avqx6il9V1aRlYiGpajbSEcbGz7fSCOJZkJ6XrYBbaiaZrEYO8W/Hq2pOvKB2x0RjPfkrTCVl0+bcPhVJW7MUiCsKGIh7jEWxlchF89+NE+BEraV+ibnTmquDgqgBXBTrRwFXxYpLBkWfqi6uCV4y/fMZIleloSzzaQABK18Hotxn9+aMbSmlH3VW6vwgBEQJ2Ih3jVCsrs0EOjZAkGkZIYoTkOEZIatIExsXOFeaDgVE5wKjXO1Dy/wEu5U1ouX4DAA=="""
PILOT_GZIP_B64 = """H4sIABRsv2oC/61a247bNhB977e4AC+SZT+2sNMG2EuQeIP2iWAkWhZWJlVK2jT5+lLuer2+SBqO+LQGFsPLXM45M1QqayWKbLa15qfSorIqK9KmMHqWK62sbFQm1L+VSt2P2V426U7V4vSvUn5T5UyLbaGzQuf1L1/++IuL3+8+PqzEtjQmE5mqmp0glM4+rz88bdart78b26oZ7bXg0W0L9t5CGbEvxYssi0x2pxYkWg5u9Of6bvX4tPn1w2/3H+/+Fq0uGpEVe6Xrznpf1Ic7CkLY7NPnx6/rh+OfwyLk+rRW7U2jRN2toHN30QRw0VLqLDUvyop0J3WunBkH++dyRxL73LhOXfBEt3cpK1Eq+Sy7/ckSd2wKNKtdhJTYGqtSWTduPz4Y3dX66/ru8dP9+mEjtm3TWpek2lnvX6M8GJ73xqlL09zYIpWlW6FRtjLlcQ0OXKPVdVtVxnYJb1VtyvawQC6LbhUCTJTXtI7wgaYLgOl1RdAYFdqIgSNUV24r5+C6Kl05nXJq3DU3TjsUlYtcduV68Exl3cm11Gm3KYXbv0+OvZJayDy3Kj/l2JDfRlN0jnI7iwGwd6uiGAMYniPsctDiCipTZRuX882PN8d3sOEfYs5QecwJyozR0SPewLUFOPmH0WHhg83V7kd9SEZ7OMVLccKqJRCr+kpxDnLCVU5RcE5dsRIHV883t7PIbZGd828AUotxGcMCBQ0OIM60LaUV0irpe/n+A1ACzJqbESB4oqLgizvoTtX/YFKqF1WOJk4vbsKdrY2LtBQ/lTUjJDVyUm9N68rw6eF1n7cfUK9yOo2PErzCxCHvBKkTzVHin8/9xP8tRuN+eo4wNETSaDL/uiVCaNgogJqOkA2TEzwoaopRGeKIHSwP+5tDjscMhhBMEUFB4oSKZ7iK94CoC8ZLUEnQV+/co94pNPdL0zp+lPXzmTGu/WWLwYP3G1KCclQE7SrOM4gn2PaegIjuRtdJUQcdoZrLvrEtm6LLdWNF4+r7fWlHiPpkCTjt+/Q5VN93Y7fXXrUWMk1bd5fTPAM1HoAKoJuiYgmH0wGdypHcEcXTJFEY9oSD3gB/Ugb3ZKP27jBvWXTsezrdPZ3H53jRRsI0zZRPV3Fj+ANv4pLpcfHohVdf7rvKXm3uR1oTkDoMkFMj2n+0xAhi+sIW04bDCxRHUi8xcQDiut2L70WzM60Tiq097h97jDAHqGjh21728VD/CJBFAcYsFDfPG3kpudjbbdocKtSZO69jJ0U3R8ZjYHFO4CptujqxplSj4A/CqgWiRCLvUCOnrhFc2wyDepiioATbYcR+fT1lPs15X2XgZ2jR0u+8zOvdsxft5yH0nMfo8PgqRsNgAQkixUiI2XOEH7clQUIZY949WCAJF4JXJkhRAr/H+TSYoiYtHD58HmrgsCM0ylBNM4mnPdFQP4TinpPVKA7QzXgo4AtUafWzNt+1x4RouEtcBtDiLBAXM/Br4rnYYZ6cRAJwKNj9l4fFcy9jyLEXj7CGBFXBNPFxcO/YwFMaMe93r5GXDsiAzEvlDzVnyI8PloEGjCPMCK/hCdLS/90ywbQomFHq4IMWaObhyUpsjuNOiuRqHoNL4aKBwXxqQzzc+U/rDDtXqkYeJFFp6trvg6qhvPf4rquvm0J/bNgrJP8DBIZ2cf8qAAA="""

def decode_embedded_csv(payload):
    raw = gzip.decompress(base64.b64decode(payload.encode()))
    from io import BytesIO
    return pd.read_csv(BytesIO(raw))

blind_cases = decode_embedded_csv(REVIEWER_PACK_GZIP_B64)
frozen_reference = decode_embedded_csv(PILOT_GZIP_B64)

blind_cases = blind_cases.head(MAX_CASES).copy()

MODEL_INPUT_COLUMNS = [
    "review_order",
    "case_id",
    "workflow_intent",
    "artifacts_json",
    "steps_json"
]
blind_cases = blind_cases[MODEL_INPUT_COLUMNS].copy()

print("Blinded cases:", len(blind_cases))
print("Unique case IDs:", blind_cases.case_id.nunique())
display(blind_cases.head(3))

# Part II — Frozen v3 reference record

This is the verifier freeze record created before the multi-model evaluation.

In [ ]:
FROZEN_V3_RECORD = {
  "frozen_at_utc": "2026-10-02T08:19:38.747739+00:00",
  "verifier_version": "SciGeoGuard-X-v3",
  "v3_notebook": "/mnt/data/SciGeoGuard_X_Publication_Experiments_v3.ipynb",
  "v3_results_zip": "/mnt/data/SciGeoGuard_X_v3_All_Results.zip",
  "v3_notebook_sha256": "afdb1ad9c2cb8742b3d26638067e80fb8619afd6aa019a58769e5acbdc2c15d2",
  "v3_results_zip_sha256": "8cb3f200453e8018f5afd0ce3c95130f6efdee4733f819727b5b16fc159ef992",
  "freeze_status": "VERIFIER_NOTEBOOK_HASHED"
}

print(json.dumps(FROZEN_V3_RECORD, indent=2))
assert FROZEN_V3_RECORD["freeze_status"] == "VERIFIER_NOTEBOOK_HASHED"

# Part III — Install local-model dependencies

No external inference API is used. The notebook attempts FP16 model loading first and falls back to 4-bit loading on GPU memory exhaustion.

In [ ]:
if RUN_MODELS:
    import sys, subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.45,<5",
        "accelerate>=0.34",
        "bitsandbytes>=0.43",
        "huggingface_hub>=0.25",
        "sentencepiece"
    ])

    import torch

    if not torch.cuda.is_available():
        raise RuntimeError(
            "No CUDA GPU detected. In Google Colab choose "
            "Runtime -> Change runtime type -> T4 GPU."
        )

    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("RUN_MODELS=False: dependency installation skipped.")

# Part IV — Reproducible model registry

At run time, the notebook records the exact Hugging Face revision SHA for each judge model.

In [ ]:
MODEL_REGISTRY = {
    "judge_A": {"model_id": MODEL_A},
    "judge_B": {"model_id": MODEL_B},
    "judge_C": {"model_id": MODEL_C},
}

if RUN_MODELS:
    from huggingface_hub import model_info

    for name, rec in MODEL_REGISTRY.items():
        try:
            info = model_info(rec["model_id"])
            rec["revision_sha"] = info.sha
        except Exception as e:
            rec["revision_sha"] = None
            rec["revision_lookup_error"] = str(e)

registry_path = WORK / "judge_model_registry.json"
registry_path.write_text(
    json.dumps(MODEL_REGISTRY, indent=2),
    encoding="utf-8"
)

MODEL_REGISTRY

# Part V — Blind scientific-review rubric

Every judge receives the same rubric and the same case content. No generated truth or SciGeoGuard-X output is included.

In [ ]:
SYSTEM_PROMPT = """
You are an independent scientific reviewer for Earth-observation and geospatial workflows.

Judge only from the scientific goal, artifact metadata, and operations supplied.
Do not infer a hidden expected answer.
Do not assume missing metadata.

Definitions:
PROVEN:
The supplied evidence is sufficient to support the stated scientific workflow for its declared purpose.

REFUTED:
At least one supplied operation, physical assumption, geospatial transformation, temporal relationship,
measurement interpretation, or data-semantic choice is scientifically incompatible with the stated goal.

UNPROVEN:
The workflow might be scientifically valid, but necessary metadata or evidence are missing, so validity
cannot be established.

Important principles:
- Code execution success does not imply scientific validity.
- Categorical and continuous data have different resampling semantics.
- Units, CRS, spatial support, temporal support, acquisition times, vertical datums, processing level,
  quality masks, provenance, uncertainty, and train/test independence can matter.
- Do not invent facts that are absent from the case.

Return exactly one JSON object with:
{
  "verdict": "PROVEN" or "REFUTED" or "UNPROVEN",
  "error_class": "short category or NONE",
  "severity": "LOW" or "MEDIUM" or "HIGH" or "CRITICAL",
  "confidence": integer 1 to 5,
  "reason": "concise scientific reason"
}
""".strip()

def make_case_prompt(row):
    return f"""
Scientific goal / intent:
{row['workflow_intent']}

Artifacts:
{row['artifacts_json']}

Workflow operations:
{row['steps_json']}

Evaluate this case independently according to the rubric.
""".strip()

print(SYSTEM_PROMPT[:1000])

# Part VI — Robust output parser

Malformed judge output becomes `UNPARSED`; it is never silently converted into a scientific verdict.

In [ ]:
VALID_VERDICTS = {"PROVEN", "REFUTED", "UNPROVEN"}
VALID_SEVERITIES = {"LOW", "MEDIUM", "HIGH", "CRITICAL"}

def extract_first_json_object(text):
    if not isinstance(text, str):
        return None

    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.I)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        obj = json.loads(cleaned)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    starts = [m.start() for m in re.finditer(r"\{", cleaned)]

    for start in starts:
        depth = 0
        in_string = False
        escape = False

        for i in range(start, len(cleaned)):
            ch = cleaned[i]

            if escape:
                escape = False
                continue

            if ch == "\\" and in_string:
                escape = True
                continue

            if ch == '"':
                in_string = not in_string
                continue

            if in_string:
                continue

            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1

                if depth == 0:
                    candidate = cleaned[start:i+1]
                    try:
                        obj = json.loads(candidate)
                        if isinstance(obj, dict):
                            return obj
                    except Exception:
                        break

    return None

def parse_judge_output(text):
    obj = extract_first_json_object(text)

    verdict = "UNPARSED"
    error_class = ""
    severity = ""
    confidence = np.nan
    reason = ""

    if obj:
        v = str(obj.get("verdict", "")).strip().upper()
        if v in VALID_VERDICTS:
            verdict = v

        error_class = str(obj.get("error_class", "")).strip()

        s = str(obj.get("severity", "")).strip().upper()
        if s in VALID_SEVERITIES:
            severity = s

        try:
            c = int(obj.get("confidence"))
            if 1 <= c <= 5:
                confidence = c
        except Exception:
            pass

        reason = str(obj.get("reason", "")).strip()

    if verdict == "UNPARSED":
        m = re.search(
            r"(?:verdict|decision)\s*[:=]\s*[\"']?(PROVEN|REFUTED|UNPROVEN)",
            text,
            flags=re.I
        )
        if m:
            verdict = m.group(1).upper()

    return {
        "verdict": verdict,
        "error_class": error_class,
        "severity": severity,
        "confidence": confidence,
        "reason": reason,
        "raw_output": text
    }

tests = [
    '{"verdict":"REFUTED","error_class":"CRS","severity":"HIGH","confidence":5,"reason":"Planar area in EPSG:4326."}',
    '```json\\n{"verdict":"UNPROVEN","error_class":"provenance","severity":"MEDIUM","confidence":4,"reason":"Source missing."}\\n```'
]

for t in tests:
    print(parse_judge_output(t))

# Part VII — Sequential model loading

In [ ]:
def load_judge_model(model_id):
    import torch
    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM,
        BitsAndBytesConfig,
    )

    print("Loading:", model_id)

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True
    )

    common = dict(
        pretrained_model_name_or_path=model_id,
        trust_remote_code=True,
        device_map="auto",
        low_cpu_mem_usage=True
    )

    try:
        model = AutoModelForCausalLM.from_pretrained(
            **common,
            torch_dtype=torch.float16
        )
        load_mode = "fp16"

    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise

        print("FP16 OOM; retrying in 4-bit.")
        gc.collect()
        torch.cuda.empty_cache()

        quant = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )

        model = AutoModelForCausalLM.from_pretrained(
            **common,
            quantization_config=quant
        )
        load_mode = "4bit_nf4"

    model.eval()
    return tokenizer, model, load_mode

def unload_judge(tokenizer, model):
    import torch
    del tokenizer
    del model
    gc.collect()
    torch.cuda.empty_cache()

# Part VIII — Deterministic inference

In [ ]:
def generate_judgment(tokenizer, model, row):
    import torch

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_case_prompt(row)}
    ]

    try:
        rendered = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    except Exception:
        rendered = (
            "SYSTEM:\n" + SYSTEM_PROMPT +
            "\n\nUSER:\n" + make_case_prompt(row) +
            "\n\nASSISTANT:\n"
        )

    inputs = tokenizer(
        rendered,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

# Part IX — Resumable blind-judge runner

In [ ]:
def run_judge(
    judge_name,
    model_id,
    cases,
    order_seed,
    checkpoint_every=CHECKPOINT_EVERY
):
    out_path = WORK / f"{judge_name}_predictions.csv"

    if out_path.exists():
        existing = pd.read_csv(out_path)
        done = set(existing.case_id.astype(str))
        rows = existing.to_dict("records")
        print(judge_name, "resuming from", len(done), "completed cases")
    else:
        done = set()
        rows = []

    order = cases.sample(
        frac=1,
        random_state=order_seed
    ).reset_index(drop=True)

    remaining = order[
        ~order.case_id.astype(str).isin(done)
    ].copy()

    if len(remaining) == 0:
        print(judge_name, "already complete.")
        return pd.DataFrame(rows)

    tokenizer, model, load_mode = load_judge_model(model_id)

    try:
        for k, (_, case) in enumerate(remaining.iterrows(), start=1):
            try:
                raw = generate_judgment(tokenizer, model, case)
                parsed = parse_judge_output(raw)

                rec = {
                    "case_id": case.case_id,
                    "judge": judge_name,
                    "model_id": model_id,
                    "load_mode": load_mode,
                    **parsed
                }

            except Exception as e:
                rec = {
                    "case_id": case.case_id,
                    "judge": judge_name,
                    "model_id": model_id,
                    "load_mode": load_mode,
                    "verdict": "UNPARSED",
                    "error_class": "",
                    "severity": "",
                    "confidence": np.nan,
                    "reason": "",
                    "raw_output": "",
                    "inference_error": repr(e)
                }

            rows.append(rec)

            if k % checkpoint_every == 0 or k == len(remaining):
                pd.DataFrame(rows).drop_duplicates(
                    subset=["case_id"],
                    keep="last"
                ).to_csv(out_path, index=False)

                parsed_n = sum(
                    r.get("verdict") in VALID_VERDICTS
                    for r in rows
                )

                print(
                    f"{judge_name}: "
                    f"{len(rows)}/{len(order)} saved; "
                    f"parsed={parsed_n}"
                )

    finally:
        unload_judge(tokenizer, model)

    return pd.read_csv(out_path)

# Part X — Run Judge A and Judge B independently

In [ ]:
if RUN_MODELS:
    judge_A = run_judge(
        "judge_A",
        MODEL_A,
        blind_cases,
        order_seed=101
    )

    judge_B = run_judge(
        "judge_B",
        MODEL_B,
        blind_cases,
        order_seed=202
    )
else:
    judge_A = pd.DataFrame()
    judge_B = pd.DataFrame()
    print("Model execution skipped.")

# Part XI — A/B agreement before adjudication

In [ ]:
def cohens_kappa(a, b, labels=("PROVEN","REFUTED","UNPROVEN")):
    a = pd.Series(a)
    b = pd.Series(b)

    ok = a.isin(labels) & b.isin(labels)
    a = a[ok].reset_index(drop=True)
    b = b[ok].reset_index(drop=True)

    if len(a) == 0:
        return np.nan

    po = float((a == b).mean())

    pa = a.value_counts(normalize=True)
    pb = b.value_counts(normalize=True)

    pe = sum(
        float(pa.get(l, 0)) * float(pb.get(l, 0))
        for l in labels
    )

    if math.isclose(1 - pe, 0):
        return np.nan

    return (po - pe) / (1 - pe)

AB = pd.DataFrame()
ab_summary = {}

if not judge_A.empty and not judge_B.empty:
    AB = (
        blind_cases
        .merge(
            judge_A[["case_id","verdict","reason","confidence"]]
            .rename(columns={
                "verdict":"A_verdict",
                "reason":"A_reason",
                "confidence":"A_confidence"
            }),
            on="case_id"
        )
        .merge(
            judge_B[["case_id","verdict","reason","confidence"]]
            .rename(columns={
                "verdict":"B_verdict",
                "reason":"B_reason",
                "confidence":"B_confidence"
            }),
            on="case_id"
        )
    )

    valid = AB[
        AB.A_verdict.isin(VALID_VERDICTS) &
        AB.B_verdict.isin(VALID_VERDICTS)
    ].copy()

    ab_summary = {
        "cases_scored_by_both": int(len(valid)),
        "A_parse_success": float(judge_A.verdict.isin(VALID_VERDICTS).mean()),
        "B_parse_success": float(judge_B.verdict.isin(VALID_VERDICTS).mean()),
        "raw_A_B_agreement": float(
            (valid.A_verdict == valid.B_verdict).mean()
        ) if len(valid) else np.nan,
        "cohens_kappa_A_B": float(
            cohens_kappa(valid.A_verdict, valid.B_verdict)
        ) if len(valid) else np.nan,
        "A_B_disagreements": int(
            (valid.A_verdict != valid.B_verdict).sum()
        )
    }

    print(json.dumps(ab_summary, indent=2))
else:
    print("A/B agreement awaits model execution.")

# Part XII — Blind Judge C adjudication

Judge C receives only the original case, never A or B's answers.

In [ ]:
def cases_needing_C(ab):
    if ab.empty:
        return pd.DataFrame(columns=blind_cases.columns)

    mask = (
        (~ab.A_verdict.isin(VALID_VERDICTS)) |
        (~ab.B_verdict.isin(VALID_VERDICTS)) |
        (ab.A_verdict != ab.B_verdict)
    )

    ids = set(ab.loc[mask, "case_id"])

    return blind_cases[
        blind_cases.case_id.isin(ids)
    ].copy()

judge_C_cases = cases_needing_C(AB)

print("Cases requiring Judge C:", len(judge_C_cases))

if RUN_MODELS and len(judge_C_cases):
    judge_C = run_judge(
        "judge_C",
        MODEL_C,
        judge_C_cases,
        order_seed=303
    )
else:
    judge_C = pd.DataFrame()

# Part XIII — Multi-model consensus

In [ ]:
def majority_consensus(a, b, c=None):
    vals = [x for x in [a,b,c] if x in VALID_VERDICTS]

    if not vals:
        return "UNRESOLVED", "NO_VALID_JUDGE_OUTPUT"

    counts = pd.Series(vals).value_counts()

    if counts.iloc[0] >= 2:
        return counts.index[0], "MAJORITY"

    return "UNRESOLVED", "NO_MAJORITY"

consensus = pd.DataFrame()

if not AB.empty:
    consensus = AB.copy()

    if not judge_C.empty:
        c_small = judge_C[[
            "case_id","verdict","reason","confidence"
        ]].rename(columns={
            "verdict":"C_verdict",
            "reason":"C_reason",
            "confidence":"C_confidence"
        })

        consensus = consensus.merge(
            c_small,
            on="case_id",
            how="left"
        )
    else:
        consensus["C_verdict"] = np.nan
        consensus["C_reason"] = np.nan
        consensus["C_confidence"] = np.nan

    final = consensus.apply(
        lambda r: majority_consensus(
            r.A_verdict,
            r.B_verdict,
            r.C_verdict
        ),
        axis=1
    )

    consensus["consensus_verdict"] = [x[0] for x in final]
    consensus["consensus_rule"] = [x[1] for x in final]

    print(consensus.consensus_verdict.value_counts(dropna=False))
else:
    print("Consensus awaits model execution.")

# Part XIV — Compare model consensus with frozen SciGeoGuard-X and generated labels

Terminology is kept explicit:

- `generated_expected`: v3 generator label;
- `frozen_prediction`: frozen SciGeoGuard-X v3 output;
- `consensus_verdict`: blind LLM majority result.

In [ ]:
evaluation = pd.DataFrame()

if not consensus.empty:
    evaluation = consensus.merge(
        frozen_reference[[
            "case_id",
            "frozen_prediction",
            "generated_expected"
        ]],
        on="case_id",
        how="left"
    )

    evaluation["A_equals_generated"] = (
        evaluation.A_verdict == evaluation.generated_expected
    )

    evaluation["B_equals_generated"] = (
        evaluation.B_verdict == evaluation.generated_expected
    )

    evaluation["consensus_equals_generated"] = (
        evaluation.consensus_verdict == evaluation.generated_expected
    )

    evaluation["scigeoguard_equals_consensus"] = (
        evaluation.frozen_prediction == evaluation.consensus_verdict
    )

    display(
        evaluation[[
            "case_id",
            "A_verdict",
            "B_verdict",
            "C_verdict",
            "consensus_verdict",
            "frozen_prediction",
            "generated_expected"
        ]].head(10)
    )
else:
    print("Evaluation awaits model execution.")

# Part XV — Metrics

In [ ]:
def exact_accuracy(truth, pred):
    t = pd.Series(truth)
    p = pd.Series(pred)
    ok = t.isin(VALID_VERDICTS) & p.isin(VALID_VERDICTS)

    if not ok.any():
        return np.nan

    return float((t[ok].values == p[ok].values).mean())

def fscr(truth, pred):
    t = pd.Series(truth)
    p = pd.Series(pred)
    invalid = t.eq("REFUTED")

    if not invalid.any():
        return np.nan

    return float((p[invalid] == "PROVEN").mean())

metrics = {}

if not evaluation.empty:
    resolved = evaluation.consensus_verdict.isin(VALID_VERDICTS)

    metrics = {
        **ab_summary,
        "C_cases_requested": int(len(judge_C_cases)),
        "C_parse_success": (
            float(judge_C.verdict.isin(VALID_VERDICTS).mean())
            if not judge_C.empty else np.nan
        ),
        "consensus_coverage": float(resolved.mean()),
        "unresolved_cases": int((~resolved).sum()),
        "A_vs_generated_accuracy": exact_accuracy(
            evaluation.generated_expected,
            evaluation.A_verdict
        ),
        "B_vs_generated_accuracy": exact_accuracy(
            evaluation.generated_expected,
            evaluation.B_verdict
        ),
        "consensus_vs_generated_accuracy": exact_accuracy(
            evaluation.generated_expected,
            evaluation.consensus_verdict
        ),
        "frozen_scigeoguard_vs_consensus_accuracy": exact_accuracy(
            evaluation.consensus_verdict,
            evaluation.frozen_prediction
        ),
        "kappa_A_vs_B": float(
            cohens_kappa(
                evaluation.A_verdict,
                evaluation.B_verdict
            )
        ),
        "kappa_consensus_vs_scigeoguard": float(
            cohens_kappa(
                evaluation.consensus_verdict,
                evaluation.frozen_prediction
            )
        ),
        "A_FSCR_vs_generated_reference": fscr(
            evaluation.generated_expected,
            evaluation.A_verdict
        ),
        "B_FSCR_vs_generated_reference": fscr(
            evaluation.generated_expected,
            evaluation.B_verdict
        ),
        "consensus_FSCR_vs_generated_reference": fscr(
            evaluation.generated_expected,
            evaluation.consensus_verdict
        ),
        "scigeoguard_FSCR_vs_generated_reference": fscr(
            evaluation.generated_expected,
            evaluation.frozen_prediction
        )
    }

    metrics_df = pd.DataFrame(
        [{"metric":k,"value":v} for k,v in metrics.items()]
    )

    display(metrics_df)
else:
    metrics_df = pd.DataFrame(columns=["metric","value"])

# Part XVI — Pairwise agreement matrix

In [ ]:
agreement_matrix = pd.DataFrame()

if not evaluation.empty:
    systems = {
        "Judge_A": evaluation.A_verdict,
        "Judge_B": evaluation.B_verdict,
        "Judge_C": evaluation.C_verdict,
        "Consensus": evaluation.consensus_verdict,
        "SciGeoGuard_X": evaluation.frozen_prediction,
        "Generated_Label": evaluation.generated_expected
    }

    rows = []

    for n1,s1 in systems.items():
        for n2,s2 in systems.items():
            rows.append({
                "system_1":n1,
                "system_2":n2,
                "exact_agreement":exact_accuracy(s1,s2),
                "cohens_kappa":cohens_kappa(s1,s2)
            })

    agreement_matrix = pd.DataFrame(rows)

    display(
        agreement_matrix.pivot(
            index="system_1",
            columns="system_2",
            values="cohens_kappa"
        )
    )

# Part XVII — Confidence analysis

In [ ]:
confidence_summary = pd.DataFrame()

if not evaluation.empty:
    records = []

    for name, col_c in [
        ("Judge_A","A_confidence"),
        ("Judge_B","B_confidence"),
        ("Judge_C","C_confidence")
    ]:
        if col_c in evaluation.columns:
            x = pd.to_numeric(
                evaluation[col_c],
                errors="coerce"
            )

            records.append({
                "judge":name,
                "mean_confidence":float(x.mean()) if x.notna().any() else np.nan,
                "median_confidence":float(x.median()) if x.notna().any() else np.nan,
                "n_confidence_scores":int(x.notna().sum())
            })

    confidence_summary = pd.DataFrame(records)
    display(confidence_summary)

# Part XVIII — Targeted human audit

The human-facing CSV hides all model/SciGeoGuard/generated labels.

In [ ]:
human_audit_internal = pd.DataFrame()
human_audit_blind = pd.DataFrame()

if not evaluation.empty:
    must_audit = (
        evaluation.consensus_verdict.eq("UNRESOLVED") |
        (
            evaluation.consensus_verdict.isin(VALID_VERDICTS) &
            evaluation.frozen_prediction.ne(
                evaluation.consensus_verdict
            )
        ) |
        (
            evaluation.consensus_verdict.isin(VALID_VERDICTS) &
            evaluation.generated_expected.ne(
                evaluation.consensus_verdict
            )
        )
    )

    priority = evaluation[must_audit].copy()

    remaining = evaluation[
        ~evaluation.case_id.isin(priority.case_id)
    ].copy()

    needed = max(0, HUMAN_AUDIT_TARGET-len(priority))

    if needed and len(remaining):
        random_spots = remaining.sample(
            n=min(needed,len(remaining)),
            random_state=SEED
        )

        human_audit_internal = pd.concat(
            [priority,random_spots],
            ignore_index=True
        )
    else:
        human_audit_internal = priority.copy()

    ids = human_audit_internal.case_id.tolist()

    human_audit_blind = blind_cases[
        blind_cases.case_id.isin(ids)
    ].copy()

    human_audit_blind["human_verdict"] = ""
    human_audit_blind["human_error_class"] = ""
    human_audit_blind["human_severity"] = ""
    human_audit_blind["human_confidence_1_to_5"] = ""
    human_audit_blind["human_reason"] = ""
    human_audit_blind["reviewer_id"] = ""

    print("Human audit cases:", len(human_audit_blind))
else:
    print("Human audit sample awaits model execution.")

# Part XIX — Publication figures

In [ ]:
FIGURES = []

if not evaluation.empty:
    import matplotlib.pyplot as plt

    systems = ["Judge A","Judge B","Consensus","SciGeoGuard-X"]

    vals = [
        exact_accuracy(evaluation.generated_expected,evaluation.A_verdict),
        exact_accuracy(evaluation.generated_expected,evaluation.B_verdict),
        exact_accuracy(evaluation.generated_expected,evaluation.consensus_verdict),
        exact_accuracy(evaluation.generated_expected,evaluation.frozen_prediction)
    ]

    p = WORK/"figure_multimodel_generated_agreement.png"
    plt.figure(figsize=(8,4))
    plt.bar(systems,vals)
    plt.ylabel("Exact agreement with generated reference")
    plt.ylim(0,1.05)
    plt.xticks(rotation=20,ha="right")
    plt.tight_layout()
    plt.savefig(p,dpi=220)
    plt.show()
    FIGURES.append(p)

    vals = [
        fscr(evaluation.generated_expected,evaluation.A_verdict),
        fscr(evaluation.generated_expected,evaluation.B_verdict),
        fscr(evaluation.generated_expected,evaluation.consensus_verdict),
        fscr(evaluation.generated_expected,evaluation.frozen_prediction)
    ]

    p = WORK/"figure_multimodel_fscr.png"
    plt.figure(figsize=(8,4))
    plt.bar(systems,vals)
    plt.ylabel("FSCR vs generated reference")
    plt.ylim(0,1.05)
    plt.xticks(rotation=20,ha="right")
    plt.tight_layout()
    plt.savefig(p,dpi=220)
    plt.show()
    FIGURES.append(p)

    dist = pd.DataFrame({
        "Judge A":evaluation.A_verdict.value_counts(),
        "Judge B":evaluation.B_verdict.value_counts(),
        "Consensus":evaluation.consensus_verdict.value_counts(),
        "SciGeoGuard-X":evaluation.frozen_prediction.value_counts()
    }).fillna(0).reindex(
        ["PROVEN","REFUTED","UNPROVEN","UNRESOLVED"],
        fill_value=0
    )

    p = WORK/"figure_verdict_distribution.png"
    dist.plot(kind="bar",figsize=(9,5))
    plt.ylabel("Cases")
    plt.xlabel("Verdict")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(p,dpi=220)
    plt.show()
    FIGURES.append(p)

# Part XX — Protocol integrity checks

In [ ]:
protocol_checks = {
    "180_or_fewer_cases": len(blind_cases) <= 180,
    "unique_case_ids": blind_cases.case_id.nunique() == len(blind_cases),
    "model_input_has_no_generated_expected":
        "generated_expected" not in blind_cases.columns,
    "model_input_has_no_scigeoguard_prediction":
        "frozen_prediction" not in blind_cases.columns,
    "model_input_has_no_family":
        "family" not in blind_cases.columns,
    "judge_C_not_given_A_B_answers": True,
    "human_audit_hides_machine_labels":
        (
            human_audit_blind.empty or
            not any(
                x in human_audit_blind.columns
                for x in [
                    "A_verdict","B_verdict","C_verdict",
                    "consensus_verdict","frozen_prediction",
                    "generated_expected"
                ]
            )
        )
}

protocol_df = pd.DataFrame([
    {"check":k,"passed":bool(v)}
    for k,v in protocol_checks.items()
])

display(protocol_df)

assert protocol_df.passed.all(), "Protocol-integrity check failed."

# Part XXI — Export all results

In [ ]:
blind_cases.to_csv(
    WORK/"Blinded_180_cases.csv",
    index=False
)

frozen_reference.to_csv(
    WORK/"Frozen_v3_reference.csv",
    index=False
)

if not AB.empty:
    AB.to_csv(
        WORK/"Judge_A_B_comparison.csv",
        index=False
    )

if not consensus.empty:
    consensus.to_csv(
        WORK/"MultiModel_consensus.csv",
        index=False
    )

if not evaluation.empty:
    evaluation.to_csv(
        WORK/"SciGeoGuard_vs_MultiModel_evaluation.csv",
        index=False
    )

metrics_df.to_csv(
    WORK/"MultiModel_metrics.csv",
    index=False
)

agreement_matrix.to_csv(
    WORK/"Pairwise_agreement_matrix.csv",
    index=False
)

confidence_summary.to_csv(
    WORK/"Judge_confidence_summary.csv",
    index=False
)

protocol_df.to_csv(
    WORK/"Protocol_integrity_checks.csv",
    index=False
)

human_audit_internal.to_csv(
    WORK/"Human_Audit_INTERNAL_machine_context.csv",
    index=False
)

human_audit_blind.to_csv(
    WORK/"Human_Audit_BLIND.csv",
    index=False
)

README = f"""
SciGeoGuard-X Multi-Model Blind Validation v5
=============================================

Frozen verifier:
{json.dumps(FROZEN_V3_RECORD, indent=2)}

Judge models:
A = {MODEL_A}
B = {MODEL_B}
C = {MODEL_C}

Protocol:
- A and B independently evaluate the same blinded workflows.
- They see no generated label or SciGeoGuard-X output.
- C runs only when A/B disagree or parsing fails.
- C does not see A/B decisions.
- Consensus requires a majority.
- No-majority cases remain UNRESOLVED.
- Multi-model consensus is NOT described as human expert ground truth.
- A targeted blind human-audit CSV is exported separately.

Recommended manuscript terminology:
1. generated stress-test reference;
2. frozen SciGeoGuard-X output;
3. independent LLM Judge A;
4. independent LLM Judge B;
5. blind three-model consensus;
6. targeted human audit.

RUN_MODELS = {RUN_MODELS}
Created UTC = {datetime.now(timezone.utc).isoformat()}
"""

(WORK/"README_MULTIMODEL_VALIDATION.txt").write_text(
    README,
    encoding="utf-8"
)

zip_path = shutil.make_archive(
    str(ROOT/"SciGeoGuard_X_v5_MultiModel_Blind_Validation"),
    "zip",
    root_dir=WORK
)

print("Created:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("ZIP available at:", zip_path)

# Interpretation

A useful v5 result is not merely high agreement.

The strongest result would be:

- Judge A and Judge B show meaningful independent agreement;
- disagreements are handled by a third blind model;
- SciGeoGuard-X agrees strongly with consensus;
- remaining disagreements are interpretable;
- the targeted human audit confirms the consensus is not merely shared model bias.

The human audit remains the final anchor for claims about scientific correctness.